In [1]:
# Dependency install cell (ultimate one-shot setup for Kani + Qwen HF on SageMaker)
# Run once in a fresh kernel, then restart kernel.

import sys
import subprocess

py = sys.executable

def run(cmd):
  print(">", " ".join(cmd))
  subprocess.check_call(cmd)

# 1) Clean up known conflicting packages
run([py, "-m", "pip", "uninstall", "-y", "torchvision", "transformers", "tokenizers", "kani"])

# 2) Install a compatible set for kani HuggingEngine
run([py, "-m", "pip", "install", "--no-cache-dir", "-U",
   "kani[huggingface]==1.8.0",
   "transformers==4.51.3",
   "tokenizers==0.21.1",
   "accelerate>=0.30.0",
   "huggingface_hub>=0.30.0",
   "sentencepiece",
   "safetensors",
   "protobuf",
   "nest_asyncio",
   "python-dotenv",
   "tqdm"])

# 3) Ensure torch exists (SageMaker usually already has it)
try:
  import torch  # noqa: F401
  print("torch already available")
except Exception:
  run([py, "-m", "pip", "install", "--no-cache-dir", "-U", "torch"])

print("\nInstall complete. IMPORTANT: Restart kernel now, then run Cell 1.")

> /opt/conda/bin/python -m pip uninstall -y torchvision transformers tokenizers kani


Found existing installation: transformers 4.51.3
Uninstalling transformers-4.51.3:
  Successfully uninstalled transformers-4.51.3
Found existing installation: tokenizers 0.21.1
Uninstalling tokenizers-0.21.1:
  Successfully uninstalled tokenizers-0.21.1
Found existing installation: kani 1.8.0
Uninstalling kani-1.8.0:
  Successfully uninstalled kani-1.8.0
> /opt/conda/bin/python -m pip install --no-cache-dir -U kani[huggingface]==1.8.0 transformers==4.51.3 tokenizers==0.21.1 accelerate>=0.30.0 huggingface_hub>=0.30.0 sentencepiece safetensors protobuf nest_asyncio python-dotenv tqdm
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 151.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 192.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [transformers] [transformers]


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-multimodal 1.5.0 requires torchvision<0.25.0,>=0.21.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.


torch already available

Install complete. IMPORTANT: Restart kernel now, then run Cell 1.


In [2]:
# Quick test cell (run after kernel restart)

import importlib.metadata as md

print("python:", __import__("sys").executable)
for p in ["kani", "torch", "transformers", "accelerate", "tokenizers", "huggingface_hub"]:
  try:
      print(f"{p}: {md.version(p)}")
  except Exception:
      print(f"{p}: NOT INSTALLED")

from kani import Kani
from kani.engines.huggingface import HuggingEngine

print("Kani import OK")
print("HuggingEngine import OK")

python: /opt/conda/bin/python
kani: 1.8.0
torch: 2.6.0
transformers: 4.51.3
accelerate: 1.13.0
tokenizers: 0.21.1
huggingface_hub: 0.36.2


2026-03-12 19:01:33.281843: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773342093.297743     535 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773342093.302674     535 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-03-12 19:01:33.319153: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Kani import OK
HuggingEngine import OK


# Qwen2.5-32B-Instruct via Kani (direct pddl, direct unraveling)


In [50]:
# Cell 1: Setup imports, paths, and Qwen2.5 Kani runtime (independent)
from pathlib import Path
import os
import re
import json
import time
import asyncio
import subprocess
from datetime import datetime

from dotenv import load_dotenv
from kani import Kani
from kani.engines.huggingface import HuggingEngine

# Project root (run notebook from formalizer-scaling/src/pddl_gen)
PROJECT_ROOT = Path.cwd().parents[1]

# Put heavy caches on NVMe (recommended in SageMaker)
NVME_CACHE_ROOT = Path('/mnt/sagemaker-nvme/formalizer_cache')
HF_HOME_DIR = NVME_CACHE_ROOT / 'hf_home'
HF_HUB_CACHE_DIR = NVME_CACHE_ROOT / 'hf_hub'
TRANSFORMERS_CACHE_DIR = NVME_CACHE_ROOT / 'transformers'
TORCH_HOME_DIR = NVME_CACHE_ROOT / 'torch'
XDG_CACHE_DIR = NVME_CACHE_ROOT / 'xdg_cache'

for d in [
    NVME_CACHE_ROOT,
    HF_HOME_DIR,
    HF_HUB_CACHE_DIR,
    TRANSFORMERS_CACHE_DIR,
    TORCH_HOME_DIR,
    XDG_CACHE_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(HF_HOME_DIR)
os.environ['HUGGINGFACE_HUB_CACHE'] = str(HF_HUB_CACHE_DIR)
os.environ['TRANSFORMERS_CACHE'] = str(TRANSFORMERS_CACHE_DIR)
os.environ['TORCH_HOME'] = str(TORCH_HOME_DIR)
os.environ['XDG_CACHE_HOME'] = str(XDG_CACHE_DIR)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

DATA_DIR = PROJECT_ROOT / 'data'
PROMPT_PATH = PROJECT_ROOT / 'prompts' / 'only_pddl_instruction.txt'
DOMAIN_PDDL_PATH = DATA_DIR / 'pddl' / 'domain.pddl'
DESC_DIR = DATA_DIR / 'descriptions_unraveling'

OUTPUT_ROOT = PROJECT_ROOT / 'output' / 'direct_pddl_direct_unraveling' / 'qwen2_5_32b_instruct_kani'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

VAL_PATH = Path('/home/sagemaker-user/tools/VAL/build/linux64/Release/bin/Validate')

# Model setup (Qwen2.5 Instruct via local HuggingFace + Kani)
MODEL_NAME = 'Qwen2.5-32B-Instruct'
HF_MODEL_ID = f'Qwen/{MODEL_NAME}'

QWEN_ENGINE = HuggingEngine(
    model_id=HF_MODEL_ID,
    model_load_kwargs={
        'device_map': 'auto',
        'cache_dir': str(TRANSFORMERS_CACHE_DIR),
    },
)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('VAL exists:', VAL_PATH.exists())
print('Domain PDDL exists:', DOMAIN_PDDL_PATH.exists())
print('Prompt exists:', PROMPT_PATH.exists())
print('Model:', HF_MODEL_ID)
print('NVME cache root:', NVME_CACHE_ROOT)


PROJECT_ROOT: /home/sagemaker-user/formalizer-scaling
OUTPUT_ROOT: /home/sagemaker-user/formalizer-scaling/output/direct_pddl_direct_unraveling/gemini_3_flash
VAL exists: True
Domain PDDL exists: True
Prompt exists: True
Model: gemini-3-flash-preview


In [51]:
# Cell 2: Define target problems and load static prompt/domain text
PROBLEM_IDS = [f"p{i:02d}" if i < 100 else f"p{i}" for i in range(1, 201)]  # p01..p200

PROMPT_PREFIX = PROMPT_PATH.read_text()
DOMAIN_PDDL_TEXT = DOMAIN_PDDL_PATH.read_text()

print("Num problems:", len(PROBLEM_IDS))
print("First/last:", PROBLEM_IDS[0], PROBLEM_IDS[-1])
print("Prompt chars:", len(PROMPT_PREFIX))
print("Domain PDDL chars:", len(DOMAIN_PDDL_TEXT))

Num problems: 200
First/last: p01 p200
Prompt chars: 2544
Domain PDDL chars: 997


In [52]:
# Cell 3: Load NL descriptions for one problem ID
def load_problem_descriptions(problem_id: str):
  domain_path = DESC_DIR / f"{problem_id}_domain.txt"
  problem_path = DESC_DIR / f"{problem_id}_problem.txt"

  if not domain_path.exists():
      raise FileNotFoundError(f"Missing domain description: {domain_path}")
  if not problem_path.exists():
      raise FileNotFoundError(f"Missing problem description: {problem_path}")

  return domain_path.read_text(), problem_path.read_text()

In [53]:
# Cell 4: Build prompt for one problem (combined primer + 3-input setup)
def build_prompt(problem_id: str):
  nl_domain, nl_problem = load_problem_descriptions(problem_id)

  return (
      f"{PROMPT_PREFIX}\n\n"
      "You are an expert PDDL engineer.\n"
      "Create a syntactically correct and semantically consistent problem file "
      "using the information below.\n\n"
      "You are given three inputs:\n"
      "1) Original natural language domain description\n"
      "2) Original natural language problem description\n"
      "3) Ground-truth domain PDDL (fixed reference)\n\n"
      "### Original domain description\n"
      f"{nl_domain}\n\n"
      "### Original problem description\n"
      f"{nl_problem}\n\n"
      "### Ground-truth domain PDDL\n"
      f"{DOMAIN_PDDL_TEXT}\n\n"
      "Output contract:\n"
      "- Generate ONLY the final problem PDDL.\n"
      "- Wrap it in exactly one <problem_file>...</problem_file> block.\n"
      "- Do not output any <domain_file> block.\n"
      "- Do not include any extra text before or after <problem_file>...</problem_file>."
  )

In [54]:
# Cell 5: Call Qwen2.5 via Kani for one prompt
async def _call_qwen_async(prompt: str, temperature: float = 0.4, max_new_tokens: int = 32768):
    ai = Kani(QWEN_ENGINE, system_prompt='')
    return await ai.chat_round_str(
        prompt,
        temperature=temperature,
        max_new_tokens=max_new_tokens,
    )


def call_qwen(prompt: str, temperature: float = 0.4, max_output_tokens: int = 32768):
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(_call_qwen_async(prompt, temperature=temperature, max_new_tokens=max_output_tokens))

    import nest_asyncio
    nest_asyncio.apply()
    return loop.run_until_complete(_call_qwen_async(prompt, temperature=temperature, max_new_tokens=max_output_tokens))


In [55]:
# Cell 6: Parse model output and extract problem_file block
def extract_problem_pddl(raw_text: str):
  if not raw_text:
      return ""

  matches = re.findall(r"<problem_file>(.*?)</problem_file>", raw_text, flags=re.DOTALL | re.IGNORECASE)
  if matches:
      return matches[-1].strip()

  return ""

In [31]:
# Cell 7: Solve one generated problem via planning.domains API (online solver)
import requests
import re

def run_solver(domain_pddl_path: Path, problem_pddl_path: Path, work_dir: Path, timeout_sec: int = 300):
  solver_name = "dual-bfws-ffparser"
  base_url = "https://solver.planning.domains:5001"

  domain_txt = domain_pddl_path.read_text()
  problem_txt = problem_pddl_path.read_text()

  print(f"[solver-online] submit: {solver_name}")
  try:
      submit = requests.post(
          f"{base_url}/package/{solver_name}/solve",
          json={"domain": domain_txt, "problem": problem_txt},
          timeout=300,
      )
      submit.raise_for_status()
      job = submit.json()
  except Exception as e:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": f"submit failed: {e}",
          "stdout": "",
          "stderr": "",
      }

  result_path = job.get("result")
  if not result_path:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": f"unexpected submit response: {job}",
          "stdout": "",
          "stderr": "",
      }

  result_url = f"{base_url}{result_path}"
  start = time.time()

  while True:
      if time.time() - start > timeout_sec:
          return {
              "plan_found": False,
              "plan_text": "",
              "solver_error": f"timeout after {timeout_sec}s",
              "stdout": "",
              "stderr": "",
          }

      try:
          poll = requests.post(result_url, timeout=30)
          poll.raise_for_status()
          res = poll.json()
      except Exception as e:
          return {
              "plan_found": False,
              "plan_text": "",
              "solver_error": f"poll failed: {e}",
              "stdout": "",
              "stderr": "",
          }

      status = res.get("status")
      if status == "PENDING":
          print("[solver-online] pending...")
          time.sleep(0.5)
          continue
      break

  if "Error" in res:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": str(res.get("Error")),
          "stdout": "",
          "stderr": "",
      }

  result = res.get("result", {})
  stdout = result.get("stdout", "") if isinstance(result, dict) else ""
  stderr = result.get("stderr", "") if isinstance(result, dict) else ""
  output = result.get("output", {}) if isinstance(result, dict) else {}

  plan_text = ""
  if isinstance(output, dict):
      plan_text = output.get("plan", "") or ""
  elif isinstance(output, str):
      plan_text = output

  if isinstance(plan_text, list):
      plan_text = "\n".join(str(x) for x in plan_text)

  plan_text = str(plan_text).strip()

  # OR logic:
  # 1) explicit plan text exists
  # 2) solver stdout says a plan was found (covers cost-0 empty-plan case)
  stdout_plan_found = bool(re.search(r"Plan found with cost:\s*\d+", stdout))
  plan_found = bool(plan_text) or stdout_plan_found

  if plan_found:
      if not plan_text:
          # cost-0 case: keep empty plan file so VAL can still run
          (work_dir / "plan").write_text("")
      else:
          (work_dir / "plan").write_text(plan_text + ("\n" if not plan_text.endswith("\n") else ""))

      print("[solver-online] plan found.")
      return {
          "plan_found": True,
          "plan_text": plan_text,  # can be empty for cost-0
          "solver_error": "",
          "stdout": stdout,
          "stderr": stderr,
      }

  err = (stderr + "\n" + stdout).strip() or "plan not found"
  print("[solver-online] no plan.")
  return {
      "plan_found": False,
      "plan_text": "",
      "solver_error": err,
      "stdout": stdout,
      "stderr": stderr,
  }

In [ ]:
# Cell 7b: Solve one generated problem via planning.domains API (EC2 self-hosted endpoint, resilient polling)
import requests
import re
from requests.exceptions import ReadTimeout, ConnectionError, ChunkedEncodingError

def run_solver(domain_pddl_path: Path, problem_pddl_path: Path, work_dir: Path, timeout_sec: int = 1200):
  solver_name = "dual-bfws-ffparser"
  base_url = "http://100.51.78.30:5001"

  domain_txt = domain_pddl_path.read_text()
  problem_txt = problem_pddl_path.read_text()

  print(f"[solver-online] submit: {solver_name}")
  try:
      submit = requests.post(
          f"{base_url}/package/{solver_name}/solve",
          json={"domain": domain_txt, "problem": problem_txt},
          timeout=(10, 120),
      )
      submit.raise_for_status()
      job = submit.json()
  except Exception as e:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": f"submit failed: {e}",
          "stdout": "",
          "stderr": "",
      }

  result_path = job.get("result")
  if not result_path:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": f"unexpected submit response: {job}",
          "stdout": "",
          "stderr": "",
      }

  result_url = f"{base_url}{result_path}"
  start = time.time()

  while True:
      if time.time() - start > timeout_sec:
          return {
              "plan_found": False,
              "plan_text": "",
              "solver_error": f"timeout after {timeout_sec}s",
              "stdout": "",
              "stderr": "",
          }

      try:
          poll = requests.post(result_url, timeout=(5, 180))
          poll.raise_for_status()
          res = poll.json()
      except (ReadTimeout, ConnectionError, ChunkedEncodingError, ConnectionResetError):
          print("[solver-online] transient poll error; retrying...")
          time.sleep(1.0)
          continue
      except Exception as e:
          return {
              "plan_found": False,
              "plan_text": "",
              "solver_error": f"poll failed: {e}",
              "stdout": "",
              "stderr": "",
          }

      status = res.get("status")
      if status == "PENDING":
          print("[solver-online] pending...")
          time.sleep(0.5)
          continue
      break

  if "Error" in res:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": str(res.get("Error")),
          "stdout": "",
          "stderr": "",
      }

  result = res.get("result", {})
  stdout = result.get("stdout", "") if isinstance(result, dict) else ""
  stderr = result.get("stderr", "") if isinstance(result, dict) else ""
  output = result.get("output", {}) if isinstance(result, dict) else {}

  plan_text = ""
  if isinstance(output, dict):
      plan_text = output.get("plan", "") or ""
  elif isinstance(output, str):
      plan_text = output

  if isinstance(plan_text, list):
      plan_text = "\n".join(str(x) for x in plan_text)

  plan_text = str(plan_text).strip()

  # OR logic:
  # 1) explicit plan text exists
  # 2) solver stdout says a plan was found (covers cost-0 empty-plan case)
  stdout_plan_found = bool(re.search(r"Plan found with cost:\s*\d+", stdout))
  plan_found = bool(plan_text) or stdout_plan_found

  if plan_found:
      if not plan_text:
          # cost-0 case: keep empty plan file so VAL can still run
          (work_dir / "plan").write_text("")
      else:
          (work_dir / "plan").write_text(plan_text + ("" if plan_text.endswith("\n") else "\n"))

      print("[solver-online] plan found.")
      return {
          "plan_found": True,
          "plan_text": plan_text,  # can be empty for cost-0
          "solver_error": "",
          "stdout": stdout,
          "stderr": stderr,
      }

  err = (stderr + "\n" + stdout).strip() or "plan not found"
  print("[solver-online] no plan.")
  return {
      "plan_found": False,
      "plan_text": "",
      "solver_error": err,
      "stdout": stdout,
      "stderr": stderr,
  }

In [56]:
# Cell 7c: Online solver (public endpoint) + raw API JSON persistence
import requests
import re
import json

def run_solver(domain_pddl_path: Path, problem_pddl_path: Path, work_dir: Path, timeout_sec: int = 300):
  solver_name = "dual-bfws-ffparser"
  base_url = "https://solver.planning.domains:5001"

  domain_txt = domain_pddl_path.read_text()
  problem_txt = problem_pddl_path.read_text()

  print(f"[solver-online] submit: {solver_name}")
  try:
      submit = requests.post(
          f"{base_url}/package/{solver_name}/solve",
          json={"domain": domain_txt, "problem": problem_txt},
          timeout=300,
      )
      submit.raise_for_status()
      job = submit.json()
  except Exception as e:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": f"submit failed: {e}",
          "stdout": "",
          "stderr": "",
      }

  result_path = job.get("result")
  if not result_path:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": f"unexpected submit response: {job}",
          "stdout": "",
          "stderr": "",
      }

  result_url = f"{base_url}{result_path}"
  start = time.time()
  res = None

  while True:
      if time.time() - start > timeout_sec:
          return {
              "plan_found": False,
              "plan_text": "",
              "solver_error": f"timeout after {timeout_sec}s",
              "stdout": "",
              "stderr": "",
          }

      try:
          poll = requests.post(result_url, timeout=30)
          poll.raise_for_status()
          res = poll.json()
      except Exception as e:
          return {
              "plan_found": False,
              "plan_text": "",
              "solver_error": f"poll failed: {e}",
              "stdout": "",
              "stderr": "",
          }

      status = res.get("status")
      if status == "PENDING":
          print("[solver-online] pending...")
          time.sleep(0.5)
          continue
      break

  # Persist full final API response for debugging/truncation analysis
  (work_dir / "solver_raw_response.json").write_text(json.dumps(res, indent=2, ensure_ascii=False))

  if "Error" in res:
      return {
          "plan_found": False,
          "plan_text": "",
          "solver_error": str(res.get("Error")),
          "stdout": "",
          "stderr": "",
      }

  result = res.get("result", {})
  stdout = result.get("stdout", "") if isinstance(result, dict) else ""
  stderr = result.get("stderr", "") if isinstance(result, dict) else ""
  output = result.get("output", {}) if isinstance(result, dict) else {}

  plan_text = ""
  if isinstance(output, dict):
      plan_text = output.get("plan", "") or ""
  elif isinstance(output, str):
      plan_text = output

  if isinstance(plan_text, list):
      plan_text = "\n".join(str(x) for x in plan_text)

  plan_text = str(plan_text).strip()

  # OR logic for zero-cost empty-plan cases
  stdout_plan_found = bool(re.search(r"Plan found with cost:\s*\d+", stdout))
  plan_found = bool(plan_text) or stdout_plan_found

  if plan_found:
      if not plan_text:
          (work_dir / "plan").write_text("")
      else:
          (work_dir / "plan").write_text(plan_text + ("\n" if not plan_text.endswith("\n") else ""))

      print("[solver-online] plan found.")
      return {
          "plan_found": True,
          "plan_text": plan_text,
          "solver_error": "",
          "stdout": stdout,
          "stderr": stderr,
      }

  err = (stderr + "\n" + stdout).strip() or "plan not found"
  print("[solver-online] no plan.")
  return {
      "plan_found": False,
      "plan_text": "",
      "solver_error": err,
      "stdout": stdout,
      "stderr": stderr,
  }

In [57]:
# Cell 8: Validate plan with VAL (terminal-equivalent capture, no timeout)
def run_val(domain_pddl_path: Path, problem_pddl_path: Path, plan_path: Path):
  if not VAL_PATH.exists():
      return {
          "val_ok": False,
          "val_output": f"VAL executable not found: {VAL_PATH}",
      }

  cmd = [str(VAL_PATH), "-v", str(domain_pddl_path), str(problem_pddl_path), str(plan_path)]
  print(f"[val] start: {' '.join(cmd)}")

  proc = subprocess.run(
      cmd,
      capture_output=True,
      text=True,
  )

  stdout = proc.stdout or ""
  stderr = proc.stderr or ""
  merged = (stdout + ("\n" + stderr if stderr else "")).strip()

  val_ok = "Plan valid" in merged
  print(f"[val] done. valid={val_ok}")
  return {
      "val_ok": val_ok,
      "val_output": merged,
  }

In [58]:
# Cell 9: Save generated artifacts for one problem
def save_problem_outputs(problem_id: str, raw_text: str, problem_pddl_text: str, solver_result: dict, val_result: dict | None):
  problem_dir = OUTPUT_ROOT / problem_id
  cand_dir = problem_dir / "cand_00"
  cand_dir.mkdir(parents=True, exist_ok=True)

  # Raw model output
  (cand_dir / "raw.txt").write_text(raw_text or "")

  # Generated problem PDDL
  (cand_dir / "problem.pddl").write_text(problem_pddl_text or "")

  # Always store fixed domain used for solve/val (for reproducibility)
  (cand_dir / "domain.pddl").write_text(DOMAIN_PDDL_TEXT)

  # Solver logs
  (cand_dir / "solver_stdout.txt").write_text(solver_result.get("stdout", "") or "")
  (cand_dir / "solver_stderr.txt").write_text(solver_result.get("stderr", "") or "")

  if solver_result.get("plan_found"):
      (cand_dir / "plan.txt").write_text(solver_result.get("plan_text", "") or "")
  else:
      (cand_dir / "solver_error.txt").write_text(solver_result.get("solver_error", "") or "")

  # VAL log (only when VAL was run)
  if val_result is not None:
      (cand_dir / "validator.log").write_text(val_result.get("val_output", "") or "")

  return cand_dir

In [59]:
# Cell 10: Run one problem through full pipeline
def process_one_problem(problem_id: str):
  prompt = build_prompt(problem_id)
  raw_text = call_qwen(prompt)

  problem_pddl_text = extract_problem_pddl(raw_text)

  # If model fails to return a tagged problem file, record and stop early
  if not problem_pddl_text.strip():
      solver_result = {
          "plan_found": False,
          "plan_text": "",
          "solver_error": "No <problem_file>...</problem_file> found in model output.",
          "stdout": "",
          "stderr": "",
      }
      save_problem_outputs(problem_id, raw_text, "", solver_result, None)
      return {
          "problem": problem_id,
          "parse_ok": False,
          "plan_found": False,
          "val_ok": False,
          "solver_error": solver_result["solver_error"],
      }

  # Write generated problem first so solver can consume it
  problem_dir = OUTPUT_ROOT / problem_id / "cand_00"
  problem_dir.mkdir(parents=True, exist_ok=True)
  gen_problem_path = problem_dir / "problem.pddl"
  gen_problem_path.write_text(problem_pddl_text)

  solver_result = run_solver(
      domain_pddl_path=DOMAIN_PDDL_PATH,
      problem_pddl_path=gen_problem_path,
      work_dir=problem_dir,
      timeout_sec=300,
  )

  val_result = None
  val_ok = False
  if solver_result["plan_found"]:
      plan_path = problem_dir / "plan"
      if not plan_path.exists():
          # fallback safety if solver wrote plan text but not file
          plan_path = problem_dir / "plan.txt"
          plan_path.write_text(solver_result["plan_text"])
      val_result = run_val(DOMAIN_PDDL_PATH, gen_problem_path, plan_path)
      val_ok = bool(val_result["val_ok"])

  # Persist final artifacts
  save_problem_outputs(problem_id, raw_text, problem_pddl_text, solver_result, val_result)

  return {
      "problem": problem_id,
      "parse_ok": True,
      "plan_found": bool(solver_result["plan_found"]),
      "val_ok": val_ok,
      "solver_error": solver_result.get("solver_error", ""),
  }

In [81]:
# Cell 11: Generate only (no solver/VAL)
smoke_problem = "p10"

prompt = build_prompt(smoke_problem)
raw_text = call_qwen(prompt)
problem_pddl_text = extract_problem_pddl(raw_text)

cand_dir = OUTPUT_ROOT / smoke_problem / "cand_00"
cand_dir.mkdir(parents=True, exist_ok=True)

(cand_dir / "raw.txt").write_text(raw_text or "")
(cand_dir / "problem.pddl").write_text(problem_pddl_text or "")
(cand_dir / "domain.pddl").write_text(DOMAIN_PDDL_TEXT)

print("Generation-only done.")
print("problem:", smoke_problem)
print("extracted_problem_pddl_chars:", len(problem_pddl_text or ""))
print("saved:", cand_dir)

Generation-only done.
problem: p10
extracted_problem_pddl_chars: 391
saved: /home/sagemaker-user/formalizer-scaling/output/direct_pddl_direct_unraveling/gemini_3_flash/p10/cand_00


In [82]:
# Cell 12: Solver+VAL only (no model call, concise output) -- VAL uses ground-truth problem file
import contextlib
import io

smoke_problem = "p10"
cand_dir = OUTPUT_ROOT / smoke_problem / "cand_00"

problem_path = cand_dir / "problem.pddl"  # LLM-generated problem for solver
if not problem_path.exists() or not problem_path.read_text().strip():
  raise FileNotFoundError(f"Missing/empty generated problem file: {problem_path}")

gt_problem_path = DATA_DIR / "pddl_unraveling" / f"{smoke_problem}.pddl"  # ground-truth problem for VAL
if not gt_problem_path.exists():
  raise FileNotFoundError(f"Missing ground-truth problem file for VAL: {gt_problem_path}")

# Suppress verbose live logs from solver in notebook output
with contextlib.redirect_stdout(io.StringIO()):
  solver_result = run_solver(
      domain_pddl_path=DOMAIN_PDDL_PATH,
      problem_pddl_path=problem_path,
      work_dir=cand_dir,
      timeout_sec=2000,
  )

# Save solver logs/results
(cand_dir / "solver_stdout.txt").write_text(solver_result.get("stdout", "") or "")
(cand_dir / "solver_stderr.txt").write_text(solver_result.get("stderr", "") or "")

produced_files = ["solver_stdout.txt", "solver_stderr.txt"]
val_result = None

if solver_result["plan_found"]:
  # keep a canonical copy as plan.txt and use it for VAL
  plan_file = cand_dir / "plan.txt"
  plan_text = solver_result.get("plan_text", "")
  (cand_dir / "plan.txt").write_text(plan_text)
  produced_files.append("plan.txt")

  # Suppress verbose live logs from VAL in notebook output
  with contextlib.redirect_stdout(io.StringIO()):
      # IMPORTANT: VAL validates against ground-truth problem file
      val_result = run_val(DOMAIN_PDDL_PATH, gt_problem_path, plan_file)

  (cand_dir / "validator.log").write_text(val_result.get("val_output", "") or "")
  produced_files.append("validator.log")
else:
  (cand_dir / "solver_error.txt").write_text(solver_result.get("solver_error", "") or "")
  produced_files.append("solver_error.txt")

summary = {
  "problem": smoke_problem,
  "plan_found": bool(solver_result["plan_found"]),
  "val_ok": bool(val_result["val_ok"]) if val_result is not None else False,
  "solver_error": solver_result.get("solver_error", ""),
}

print("Produced files:")
for name in produced_files:
  print("-", name)

if val_result is not None:
  print(f"VAL result: {'valid' if val_result.get('val_ok') else 'invalid'}")
else:
  print("VAL result: not run (solver failed)")

print("Summary:")
print(json.dumps(summary, indent=2))

Produced files:
- solver_stdout.txt
- solver_stderr.txt
- plan.txt
- validator.log
VAL result: valid
Summary:
{
  "problem": "p10",
  "plan_found": true,
  "val_ok": true,
  "solver_error": ""
}


In [15]:
# Cell 13: Batch generation-only (with tqdm), default p102..p110
from tqdm.auto import tqdm

BATCH_PROBLEMS = [f"p{i:02d}" if i < 100 else f"p{i}" for i in range(51, 61)]  # change this list for other batches

gen_results = []
for pid in tqdm(BATCH_PROBLEMS, desc="LLM generation", unit="problem"):
  try:
      prompt = build_prompt(pid)
      raw_text = call_qwen(prompt)
      problem_pddl_text = extract_problem_pddl(raw_text)

      cand_dir = OUTPUT_ROOT / pid / "cand_00"
      cand_dir.mkdir(parents=True, exist_ok=True)

      (cand_dir / "raw.txt").write_text(raw_text or "")
      (cand_dir / "problem.pddl").write_text(problem_pddl_text or "")
      (cand_dir / "domain.pddl").write_text(DOMAIN_PDDL_TEXT)

      ok = bool(problem_pddl_text.strip())
      gen_results.append({"problem": pid, "generation_ok": ok, "error": ""})
      print(f"{pid}: generation {'ok' if ok else 'failed (no <problem_file>)'}")
  except Exception as e:
      gen_results.append({"problem": pid, "generation_ok": False, "error": str(e)})
      print(f"{pid}: generation error -> {e}")

print("\nGeneration batch done.")
print("total:", len(gen_results))
print("ok:", sum(1 for r in gen_results if r['generation_ok']))
print("failed:", sum(1 for r in gen_results if not r['generation_ok']))

LLM generation:   0%|          | 0/10 [00:00<?, ?problem/s]

p51: generation ok
p52: generation ok
p53: generation ok
p54: generation ok
p55: generation ok
p56: generation ok
p57: generation ok
p58: generation ok
p59: generation ok
p60: generation ok

Generation batch done.
total: 10
ok: 10
failed: 0


In [ ]:
# Cell 14: Batch solver+VAL only (self-contained, with its own problem handle)
from tqdm.auto import tqdm
import contextlib
import io

BATCH_PROBLEMS_SOLVER_VAL = [f"p{i:03d}" for i in range(102, 111)]  # change as needed

sv_results = []
for pid in tqdm(BATCH_PROBLEMS_SOLVER_VAL, desc="Solver+VAL", unit="problem"):
  cand_dir = OUTPUT_ROOT / pid / "cand_00"
  problem_path = cand_dir / "problem.pddl"

  if not problem_path.exists() or not problem_path.read_text().strip():
      msg = f"missing/empty problem file: {problem_path}"
      sv_results.append({"problem": pid, "plan_found": False, "val_ok": False, "error": msg})
      print(f"{pid}: skipped ({msg})")
      continue

  # Suppress verbose live logs in batch mode
  with contextlib.redirect_stdout(io.StringIO()):
      solver_result = run_solver(
          domain_pddl_path=DOMAIN_PDDL_PATH,
          problem_pddl_path=problem_path,
          work_dir=cand_dir,
          timeout_sec=300,
      )

  (cand_dir / "solver_stdout.txt").write_text(solver_result.get("stdout", "") or "")
  (cand_dir / "solver_stderr.txt").write_text(solver_result.get("stderr", "") or "")

  val_ok = False
  if solver_result.get("plan_found"):
      plan_file = cand_dir / "plan"
      (cand_dir / "plan.txt").write_text(solver_result.get("plan_text", "") or "")

      with contextlib.redirect_stdout(io.StringIO()):
          val_result = run_val(DOMAIN_PDDL_PATH, problem_path, plan_file)

      (cand_dir / "validator.log").write_text(val_result.get("val_output", "") or "")
      val_ok = bool(val_result.get("val_ok", False))
      print(f"{pid}: val completed ({'valid' if val_ok else 'invalid'})")
  else:
      (cand_dir / "solver_error.txt").write_text(solver_result.get("solver_error", "") or "")
      print(f"{pid}: solver failed")

  sv_results.append({
      "problem": pid,
      "plan_found": bool(solver_result.get("plan_found", False)),
      "val_ok": val_ok,
      "error": solver_result.get("solver_error", ""),
  })

print("\nSolver+VAL batch done.")
print("total:", len(sv_results))
print("plan_found:", sum(1 for r in sv_results if r["plan_found"]))
print("val_ok:", sum(1 for r in sv_results if r["val_ok"]))
print("failed:", sum(1 for r in sv_results if not r["val_ok"]))

In [ ]:
# Cell 15: In-notebook metrics summary (no persistence)
import pandas as pd

# Define any batch you want here
SUMMARY_PROBLEMS = [f"p{i:03d}" for i in range(141, 151)]  # example: p141..p150

rows = []
for pid in SUMMARY_PROBLEMS:
  cand_dir = OUTPUT_ROOT / pid / "cand_00"

  solver_stdout_path = cand_dir / "solver_stdout.txt"
  solver_stderr_path = cand_dir / "solver_stderr.txt"
  solver_error_path = cand_dir / "solver_error.txt"
  plan_txt_path = cand_dir / "plan.txt"
  plan_path = cand_dir / "plan"
  val_log_path = cand_dir / "validator.log"

  solver_stdout = solver_stdout_path.read_text() if solver_stdout_path.exists() else ""
  solver_stderr = solver_stderr_path.read_text() if solver_stderr_path.exists() else ""
  solver_error = solver_error_path.read_text().strip() if solver_error_path.exists() else ""

  has_plan_txt = plan_txt_path.exists() and plan_txt_path.read_text().strip() != ""
  has_plan = plan_path.exists() and plan_path.read_text().strip() != ""
  plan_found = bool(has_plan_txt or has_plan)

  val_log = val_log_path.read_text() if val_log_path.exists() else ""
  val_ran = val_log_path.exists() and val_log.strip() != ""
  val_ok = "Plan valid" in val_log

  if val_ok:
      status = "plan_valid"
  elif plan_found and val_ran:
      status = "plan_invalid"
  else:
      status = "solver_failed"

  rows.append(
      {
          "problem_id": pid,
          "status": status,
          "plan_found": plan_found,
          "val_ran": val_ran,
          "val_ok": val_ok,
          "solver_error": solver_error,
          "solver_stdout_chars": len(solver_stdout),
          "solver_stderr_chars": len(solver_stderr),
          "cand_dir": str(cand_dir),
      }
  )

details_df = pd.DataFrame(rows).sort_values("problem_id").reset_index(drop=True)

total = len(details_df)
plan_found_n = int(details_df["plan_found"].sum())
val_ok_n = int(details_df["val_ok"].sum())
solver_failed_n = int((details_df["status"] == "solver_failed").sum())
plan_invalid_n = int((details_df["status"] == "plan_invalid").sum())

summary_df = pd.DataFrame(
  [
      {
          "num_problems": total,
          "plan_found_count": plan_found_n,
          "plan_found_rate": plan_found_n / total if total else 0.0,
          "val_ok_count": val_ok_n,
          "val_ok_rate": val_ok_n / total if total else 0.0,
          "solver_failed_count": solver_failed_n,
          "plan_invalid_count": plan_invalid_n,
      }
  ]
)

print("Summary:")
display(summary_df)

print("Per-problem details:")
display(details_df[["problem_id", "status", "plan_found", "val_ran", "val_ok", "solver_error"]])

# Keeps objects in memory for Cell 16 persistence
SUMMARY_DF = summary_df
DETAILS_DF = details_df
SUMMARY_PROBLEMS_USED = SUMMARY_PROBLEMS

In [ ]:
# Cell 16: Persist summary/details to CSV (run only when you want to save)
from datetime import datetime

# Optional label to distinguish runs (edit as needed)
SAVE_LABEL = "qwen2_5_32b_instruct_kani_batch"

# Where to save metric CSVs
METRIC_DIR = OUTPUT_ROOT / "_metrics"
METRIC_DIR.mkdir(parents=True, exist_ok=True)

if "SUMMARY_DF" not in globals() or "DETAILS_DF" not in globals():
  raise RuntimeError("Run Cell 15 first to populate SUMMARY_DF and DETAILS_DF.")

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
problems_tag = f"{SUMMARY_PROBLEMS_USED[0]}_to_{SUMMARY_PROBLEMS_USED[-1]}" if len(SUMMARY_PROBLEMS_USED) > 1 else SUMMARY_PROBLEMS_USED[0]

summary_out = METRIC_DIR / f"{SAVE_LABEL}_{problems_tag}_summary_{ts}.csv"
details_out = METRIC_DIR / f"{SAVE_LABEL}_{problems_tag}_details_{ts}.csv"

SUMMARY_DF.to_csv(summary_out, index=False)
DETAILS_DF.to_csv(details_out, index=False)

print("Saved:")
print("-", summary_out)
print("-", details_out)

# Parser for Problem Descriptions & Difference Checking

In [66]:
# Cell 17: Diff section setup (independent, PDDL-vs-PDDL for unraveling)
from pathlib import Path
import re
import json

PROJECT_ROOT = Path.cwd().parents[1]  # run notebook from formalizer-scaling/src/pddl_gen
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "output" / "direct_pddl_direct_unraveling" / "qwen2_5_32b_instruct_kani"

# Single-problem handle
DIFF_PROBLEM = "p03"  # edit as needed

# Input files
GT_PDDL_PATH = DATA_DIR / "pddl_unraveling" / f"{DIFF_PROBLEM}.pddl"
LLM_PDDL_PATH = OUTPUT_ROOT / DIFF_PROBLEM / "cand_00" / "problem.pddl"

# Output files
CAND_DIR = OUTPUT_ROOT / DIFF_PROBLEM / "cand_00"
CAND_DIR.mkdir(parents=True, exist_ok=True)

DIFF_JSON_OUT = CAND_DIR / "gt_vs_llm_diff.json"
DIFF_TXT_OUT = CAND_DIR / "gt_vs_llm_diff.txt"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Problem:", DIFF_PROBLEM)
print("GT PDDL exists:", GT_PDDL_PATH.exists())
print("LLM PDDL exists:", LLM_PDDL_PATH.exists())
print("Output dir:", CAND_DIR)

PROJECT_ROOT: /home/sagemaker-user/formalizer-scaling
Problem: p03
GT PDDL exists: True
LLM PDDL exists: True
Output dir: /home/sagemaker-user/formalizer-scaling/output/direct_pddl_direct_unraveling/gemini_3_flash/p03/cand_00


In [3]:
# Cell 18: Helper 1 - normalize whitespace
def _norm_space(s: str) -> str:
    return re.sub(r"\s+", " ", s.strip())

In [4]:
# Cell 19: Helper 2 - extract init/goal facts from a PDDL problem file (section-aware)
_fact_line_re = re.compile(r"\(\s*([a-zA-Z\-]+)([^()]*)\)")

def extract_section_facts_from_pddl(pddl_text: str):
    init_idx = pddl_text.find("(:init")
    goal_idx = pddl_text.find("(:goal")
    if init_idx == -1 or goal_idx == -1:
        raise ValueError("PDDL missing :init or :goal section.")

    init_text = pddl_text[init_idx:goal_idx]
    goal_text = pddl_text[goal_idx:]

    def _facts_from_section(sec: str):
        facts = []
        for m in _fact_line_re.finditer(sec):
            pred = m.group(1).strip()
            args = _norm_space(m.group(2)).split()

            if pred in {"clear", "on-table"} and len(args) == 1:
                facts.append((pred, args[0]))
            elif pred == "on" and len(args) == 2:
                facts.append((pred, args[0], args[1]))
            elif pred == "arm-empty" and len(args) == 0:
                facts.append((pred,))
        return facts

    return {
        "init_facts": _facts_from_section(init_text),
        "goal_facts": _facts_from_section(goal_text),
    }

In [5]:
# Cell 20: Helper 3 - section-aware, order-insensitive diff (GT vs LLM)
def diff_fact_sections(gt_pddl_text: str, llm_pddl_text: str):
    gt_secs = extract_section_facts_from_pddl(gt_pddl_text)
    llm_secs = extract_section_facts_from_pddl(llm_pddl_text)

    gt_init = set(gt_secs["init_facts"])
    gt_goal = set(gt_secs["goal_facts"])
    llm_init = set(llm_secs["init_facts"])
    llm_goal = set(llm_secs["goal_facts"])

    return {
        "init_missing_in_llm": sorted(gt_init - llm_init),
        "init_extra_in_llm": sorted(llm_init - gt_init),
        "goal_missing_in_llm": sorted(gt_goal - llm_goal),
        "goal_extra_in_llm": sorted(llm_goal - gt_goal),
        "init_match_count": len(gt_init & llm_init),
        "goal_match_count": len(gt_goal & llm_goal),
        "init_total_gt": len(gt_init),
        "goal_total_gt": len(gt_goal),
        "init_total_llm": len(llm_init),
        "goal_total_llm": len(llm_goal),
        "perfect_match": (
            gt_init == llm_init and
            gt_goal == llm_goal
        ),
    }

In [6]:
# Cell 21: Helper 4 - concise mismatch-only diff report text
def build_diff_report_text(problem_id: str, diff_result: dict):
    lines = []
    lines.append(f"Problem: {problem_id}")
    lines.append(f"Perfect match: {diff_result['perfect_match']}")
    lines.append("")

    lines.append(f"Init missing in LLM ({len(diff_result['init_missing_in_llm'])}):")
    for fact in diff_result["init_missing_in_llm"]:
        lines.append(f"- {fact}")

    lines.append("")
    lines.append(f"Init extra in LLM ({len(diff_result['init_extra_in_llm'])}):")
    for fact in diff_result["init_extra_in_llm"]:
        lines.append(f"- {fact}")

    lines.append("")
    lines.append(f"Goal missing in LLM ({len(diff_result['goal_missing_in_llm'])}):")
    for fact in diff_result["goal_missing_in_llm"]:
        lines.append(f"- {fact}")

    lines.append("")
    lines.append(f"Goal extra in LLM ({len(diff_result['goal_extra_in_llm'])}):")
    for fact in diff_result["goal_extra_in_llm"]:
        lines.append(f"- {fact}")

    return "\n".join(lines) + "\n"

In [67]:
# Cell 22: Diff run (single problem handle, persists outputs to pXXX/cand_00)
if not GT_PDDL_PATH.exists():
    raise FileNotFoundError(f"Missing ground-truth PDDL: {GT_PDDL_PATH}")
if not LLM_PDDL_PATH.exists():
    raise FileNotFoundError(f"Missing LLM-generated PDDL: {LLM_PDDL_PATH}")

gt_pddl_text = GT_PDDL_PATH.read_text()
llm_pddl_text = LLM_PDDL_PATH.read_text()

diff_result = diff_fact_sections(gt_pddl_text, llm_pddl_text)
diff_text = build_diff_report_text(DIFF_PROBLEM, diff_result)

DIFF_JSON_OUT.write_text(json.dumps(diff_result, indent=2))
DIFF_TXT_OUT.write_text(diff_text)

print("Saved:")
print("-", DIFF_JSON_OUT)
print("-", DIFF_TXT_OUT)
print("")
print("Mismatch summary:")
print(f"- init missing: {len(diff_result['init_missing_in_llm'])}")
print(f"- init extra:   {len(diff_result['init_extra_in_llm'])}")
print(f"- goal missing: {len(diff_result['goal_missing_in_llm'])}")
print(f"- goal extra:   {len(diff_result['goal_extra_in_llm'])}")
print(f"- perfect_match: {diff_result['perfect_match']}")

Saved:
- /home/sagemaker-user/formalizer-scaling/output/direct_pddl_direct_unraveling/gemini_3_flash/p03/cand_00/gt_vs_llm_diff.json
- /home/sagemaker-user/formalizer-scaling/output/direct_pddl_direct_unraveling/gemini_3_flash/p03/cand_00/gt_vs_llm_diff.txt

Mismatch summary:
- init missing: 0
- init extra:   0
- goal missing: 0
- goal extra:   0
- perfect_match: True
